# CoinMate ML — 분봉 학습 노트북 (Colab Free)

**목표**: 일봉이 아닌 **5분봉 1년치**로 학습 + **봇 매매 시뮬레이션 기반 라벨**(익절 +3.5% 먼저 vs 손절 -2% 먼저)로 학습하여, ML 확률이 **실제 봇 매매 승률**과 일치하게 만든다.

**산출물**: `xgb_model.pkl` (XGBoost + Isotonic calibrator 포함, 서버 `MLPredictor._load_model` 호환)

**예상 소요**:
- 데이터 다운: 3~5시간 (5분봉 1년치 × 240 코인). Drive 캐시되므로 재실행 시 빠름.
- 피처 + 라벨: 30분~1시간
- 학습 + calibration: 10~30분

**Colab Free 자원**: RAM 12GB / 디스크 100GB → 충분.

---

## 사용법
1. Colab 새 노트북에서 이 파일 업로드(또는 GitHub에서 import)
2. 위→아래 순서로 셀 실행
3. **Drive 마운트 셀에서 Google 계정 인증 필요**
4. 데이터 다운은 한 번만 (Drive에 캐시됨, 재실행 시 skip)
5. 최종 셀에서 `xgb_model.pkl` 다운로드 → 서버 `~/CoinMate-Backend/models/xgb_model.pkl`로 교체

## 0. 패키지 설치

In [ ]:
!pip install -q pyupbit xgboost scikit-learn pandas joblib pyarrow tqdm

## 1. Google Drive 마운트 (데이터/모델 캐시)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = '/content/drive/MyDrive/CoinMate_ML/data'
MODEL_DIR = '/content/drive/MyDrive/CoinMate_ML/models'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'캐시 디렉토리: {CACHE_DIR}')
print(f'모델 디렉토리: {MODEL_DIR}')

## 2. 설정

In [ ]:
# 분봉 단위 ('minute1','minute3','minute5','minute15','minute30','minute60')
TIMEFRAME = 'minute5'

# 다운로드 일수 (1년 권장, 메모리 빠듯하면 180일)
DAYS_BACK = 365

# 라벨링 forward window — 매수 후 N개 분봉 동안 익절/손절 발생 여부 시뮬레이션
# 5분봉 기준: 288개 = 1일, 144개 = 12시간
TF_BARS_PER_DAY = {'minute1': 1440, 'minute3': 480, 'minute5': 288, 'minute15': 96, 'minute30': 48, 'minute60': 24}
FORWARD_BARS = TF_BARS_PER_DAY[TIMEFRAME] * 1  # 1일

# 봇 매매 룰 (서버 trade_manager 기본값 — 변경 시 동기화)
TAKE_PROFIT_PCT = 3.5
STOP_LOSS_PCT = -2.0

# 학습 sample 추출 간격 (전체 분봉의 N개마다 sample 1개)
# 5분봉 기준: 12 = 매시간, 1 = 매분봉(과적합/메모리 위험)
SAMPLE_STEP = 12

print(f'설정 확인:\n  TF={TIMEFRAME} / DAYS={DAYS_BACK} / FORWARD_BARS={FORWARD_BARS} ({FORWARD_BARS/TF_BARS_PER_DAY[TIMEFRAME]:.1f}일)')
print(f'  TP={TAKE_PROFIT_PCT}% / SL={STOP_LOSS_PCT}% / SAMPLE_STEP={SAMPLE_STEP}')

## 3. 데이터 다운로드 함수 (resume 가능)

In [ ]:
import pyupbit
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
from tqdm.notebook import tqdm


def download_one(ticker: str, days: int = 365, interval: str = 'minute5') -> pd.DataFrame | None:
    """한 코인의 분봉 데이터를 거꾸로 페이지네이션해 days치 다운로드."""
    cache_path = f'{CACHE_DIR}/{interval}_{ticker}.parquet'
    if os.path.exists(cache_path):
        try:
            return pd.read_parquet(cache_path)
        except Exception:
            pass

    minutes_per_candle = int(interval.replace('minute', ''))
    total_needed = days * 24 * 60 // minutes_per_candle
    count_per_call = 200
    n_calls = (total_needed + count_per_call - 1) // count_per_call

    all_dfs = []
    to_time = None
    for i in range(n_calls):
        try:
            df = pyupbit.get_ohlcv(ticker, interval=interval, count=count_per_call, to=to_time)
            if df is None or len(df) == 0:
                break
            all_dfs.append(df)
            to_time = df.index[0]
            time.sleep(0.12)  # 분당 600회 한도 안전 마진
        except Exception as e:
            print(f'  ⚠️ {ticker} call {i}: {e}')
            time.sleep(1)

    if not all_dfs:
        return None

    full = pd.concat(all_dfs).sort_index()
    full = full[~full.index.duplicated(keep='first')]
    full.to_parquet(cache_path)
    return full


def list_tickers() -> list[str]:
    return pyupbit.get_tickers(fiat='KRW')


print('함수 준비 완료')

## 4. 데이터 다운로드 실행

⚠️ 첫 실행 시 3~5시간 (5분봉 1년치 × 240 코인). Drive에 캐시되므로 재실행 시 즉시 skip.

💡 도중에 끊겨도 코인별 parquet 파일이 누적되므로 안전. 다시 실행하면 받지 않은 것만 받음.

In [ ]:
tickers = list_tickers()
print(f'전체 KRW 마켓: {len(tickers)}개')

done, fail = 0, 0
for t in tqdm(tickers, desc='downloading'):
    try:
        df = download_one(t, DAYS_BACK, TIMEFRAME)
        if df is not None and len(df) > 100:
            done += 1
        else:
            fail += 1
    except Exception as e:
        print(f'{t}: {e}')
        fail += 1

print(f'\n완료: {done}개 / 실패: {fail}개')

## 5. 피처 엔지니어링 (서버 호환)

분봉 DataFrame → 피처 DataFrame.
**중요**: 이 함수는 추후 서버 `app/services/ml_predictor.py`의 `build_features`에도 **동일하게 복사**되어야 추론 시 호환됨.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """분봉 OHLCV → 30+ 피처.
    분봉/일봉 공통 사용 가능 (어떤 시간프레임이든)."""
    if df is None or len(df) < 50:
        return pd.DataFrame()

    feat = pd.DataFrame(index=df.index)
    close = df['close']
    high = df['high']
    low = df['low']
    volume = df['volume']
    open_p = df['open']

    # --- 가격 변화율 ---
    feat['return_1'] = close.pct_change(1) * 100
    feat['return_5'] = close.pct_change(5) * 100
    feat['return_10'] = close.pct_change(10) * 100
    feat['return_30'] = close.pct_change(30) * 100

    # --- 이동평균 거리 ---
    feat['ma5_gap'] = (close / close.rolling(5).mean() - 1) * 100
    feat['ma20_gap'] = (close / close.rolling(20).mean() - 1) * 100
    feat['ma50_gap'] = (close / close.rolling(50).mean() - 1) * 100

    # --- 변동성 ---
    feat['volatility_5'] = close.pct_change().rolling(5).std() * 100
    feat['volatility_20'] = close.pct_change().rolling(20).std() * 100
    feat['hl_range_pct'] = ((high - low) / close) * 100
    feat['upper_wick'] = (high - np.maximum(open_p, close)) / close * 100
    feat['lower_wick'] = (np.minimum(open_p, close) - low) / close * 100
    feat['body_size'] = (close - open_p).abs() / close * 100

    # --- 거래량 ---
    feat['volume_change'] = volume.pct_change() * 100
    feat['volume_ma_ratio_20'] = volume / volume.rolling(20).mean().replace(0, np.nan)
    feat['volume_ma_ratio_50'] = volume / volume.rolling(50).mean().replace(0, np.nan)

    # --- RSI (14) ---
    delta = close.diff()
    gain = delta.where(delta > 0, 0).ewm(alpha=1/14, min_periods=14).mean()
    loss = -delta.where(delta < 0, 0).ewm(alpha=1/14, min_periods=14).mean()
    rs = gain / loss.replace(0, 0.0001)
    feat['rsi'] = 100 - (100 / (1 + rs))
    feat['rsi_change'] = feat['rsi'].diff()

    # --- MACD (12,26,9) ---
    exp12 = close.ewm(span=12, adjust=False).mean()
    exp26 = close.ewm(span=26, adjust=False).mean()
    macd = exp12 - exp26
    signal = macd.ewm(span=9, adjust=False).mean()
    feat['macd_line'] = macd
    feat['macd_signal'] = signal
    feat['macd_hist'] = macd - signal

    # --- ATR (14) ---
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs(),
    ], axis=1).max(axis=1)
    feat['atr_ratio'] = tr.rolling(14).mean() / close * 100

    # --- Bollinger (20, 2σ) ---
    ma20 = close.rolling(20).mean()
    std20 = close.rolling(20).std()
    feat['bb_position'] = (close - ma20) / (std20 * 2 + 0.0001)
    feat['bb_width_ratio'] = (std20 * 4) / ma20 * 100

    # --- Stochastic (14, 3) ---
    low14 = low.rolling(14).min()
    high14 = high.rolling(14).max()
    feat['stoch_k'] = (close - low14) / (high14 - low14 + 0.0001) * 100
    feat['stoch_d'] = feat['stoch_k'].rolling(3).mean()

    # --- 모멘텀 ---
    feat['momentum_accel'] = feat['return_5'] - feat['return_10']
    feat['consecutive_up'] = (close > close.shift(1)).astype(int).rolling(5).sum()

    # --- 가격-거래량 상관 ---
    feat['vol_price_corr'] = close.rolling(20).corr(volume)

    # --- 시간 컨텍스트 ---
    feat['hour'] = df.index.hour
    feat['dayofweek'] = df.index.dayofweek

    return feat


# 샘플 확인
sample_ticker = 'KRW-BTC'
sample_df = pd.read_parquet(f'{CACHE_DIR}/{TIMEFRAME}_{sample_ticker}.parquet')
sample_feat = build_features(sample_df)
print(f'{sample_ticker} 피처 shape: {sample_feat.shape}, 컬럼 {len(sample_feat.columns)}개')
print('컬럼 목록:', list(sample_feat.columns))
sample_feat.tail()

## 6. 라벨 생성 (봇 매매 시뮬레이션)

각 분봉 시점 t에서 매수했다고 가정하고, **다음 FORWARD_BARS 동안**:
- 고가가 `+TAKE_PROFIT_PCT%`에 **먼저** 닿으면 → `label = 1` (익절 win)
- 저가가 `-STOP_LOSS_PCT%`에 **먼저** 닿으면 → `label = 0` (손절 loss)
- 둘 다 안 닿으면 → `label = 0` (보수)

결과: ML 확률 = "이 시점에 매수 시 익절할 확률"

In [ ]:
def make_labels(df: pd.DataFrame, forward_bars: int,
                tp_pct: float = 3.5, sl_pct: float = -2.0) -> np.ndarray:
    """forward simulation 기반 라벨 생성. -1 = invalid(마지막 forward_bars개)."""
    closes = df['close'].values
    highs = df['high'].values
    lows = df['low'].values
    n = len(closes)
    labels = np.full(n, -1, dtype=np.int8)
    tp_mult = 1 + tp_pct / 100
    sl_mult = 1 + sl_pct / 100

    for i in range(n - forward_bars):
        entry = closes[i]
        tp_thresh = entry * tp_mult
        sl_thresh = entry * sl_mult
        future_high = highs[i+1:i+1+forward_bars]
        future_low = lows[i+1:i+1+forward_bars]

        tp_mask = future_high >= tp_thresh
        sl_mask = future_low <= sl_thresh
        tp_idx = np.argmax(tp_mask) if tp_mask.any() else 10**9
        sl_idx = np.argmax(sl_mask) if sl_mask.any() else 10**9
        labels[i] = 1 if tp_idx < sl_idx else 0

    return labels


# 샘플로 확인
sample_labels = make_labels(sample_df, FORWARD_BARS, TAKE_PROFIT_PCT, STOP_LOSS_PCT)
valid = sample_labels >= 0
print(f'{sample_ticker}: 유효 라벨 {valid.sum():,}개')
print(f'  익절 비율 (label=1): {sample_labels[valid].mean()*100:.1f}%')

## 7. 전체 학습 데이터셋 구축

모든 코인 × 모든 시점(SAMPLE_STEP 마다) → 단일 X, y.

In [ ]:
all_X = []
all_y = []
skipped = 0

for t in tqdm(tickers, desc='building dataset'):
    cache_path = f'{CACHE_DIR}/{TIMEFRAME}_{t}.parquet'
    if not os.path.exists(cache_path):
        skipped += 1
        continue
    try:
        df = pd.read_parquet(cache_path)
    except Exception:
        skipped += 1
        continue
    if len(df) < 200:
        skipped += 1
        continue

    feats = build_features(df)
    labels = make_labels(df, FORWARD_BARS, TAKE_PROFIT_PCT, STOP_LOSS_PCT)

    # NaN + invalid label 마스킹
    valid_mask = feats.notna().all(axis=1).values & (labels >= 0)
    feats_v = feats[valid_mask].iloc[::SAMPLE_STEP]
    labels_v = labels[valid_mask][::SAMPLE_STEP]

    if len(feats_v) < 30:
        skipped += 1
        continue

    all_X.append(feats_v)
    all_y.append(pd.Series(labels_v, index=feats_v.index))

X = pd.concat(all_X, ignore_index=True)
y = pd.concat(all_y, ignore_index=True)

print(f'\n학습 데이터: {len(X):,} 샘플 / 피처 {X.shape[1]}개 / skip {skipped}개')
print(f'양성률 (익절 먼저 닿음): {y.mean()*100:.1f}%')
print(f'메모리: {X.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

## 8. XGBoost 학습 + Isotonic Calibration

3-way 시계열 분할: 70% 학습 / 15% 보정 / 15% 테스트

In [ ]:
from xgboost import XGBClassifier
from sklearn.isotonic import IsotonicRegression

# 시계열 순서 보장 (concat 시 인덱스 reset 했으므로 이미 시간순)
total = len(X)
train_end = int(total * 0.70)
cal_end = int(total * 0.85)
X_train, X_cal, X_test = X.iloc[:train_end], X.iloc[train_end:cal_end], X.iloc[cal_end:]
y_train, y_cal, y_test = y.iloc[:train_end], y.iloc[train_end:cal_end], y.iloc[cal_end:]

print(f'분할: 학습 {len(X_train):,} / 보정 {len(X_cal):,} / 테스트 {len(X_test):,}')

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
spw = neg / max(pos, 1)
print(f'클래스 비율: 익절 {pos:,} / 손절 {neg:,} (scale_pos_weight={spw:.2f})')

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.75,
    colsample_bytree=0.7,
    min_child_weight=10,
    gamma=0.2,
    reg_alpha=0.3,
    reg_lambda=2.0,
    scale_pos_weight=spw,
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',  # CPU. GPU 있으면 'gpu_hist' 또는 device='cuda'
    n_jobs=-1,
)

print('학습 시작...')
model.fit(X_train, y_train, eval_set=[(X_cal, y_cal)], verbose=False)
print('학습 완료')

# Isotonic calibration
raw_cal = model.predict_proba(X_cal)[:, 1]
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(raw_cal, y_cal)
print('Calibration 완료')

## 9. 평가

In [ ]:
raw_test = model.predict_proba(X_test)[:, 1]
cal_test = calibrator.transform(raw_test)

uncal_acc = ((raw_test >= 0.5) == y_test.values).mean() * 100
cal_acc = ((cal_test >= 0.5) == y_test.values).mean() * 100
actual_rate = y_test.mean() * 100

print('=' * 70)
print('테스트 결과')
print('=' * 70)
print(f'  실제 익절률: {actual_rate:.1f}%')
print(f'  미보정 평균확률: {raw_test.mean()*100:.1f}% / 보정: {cal_test.mean()*100:.1f}%')
print(f'  미보정 정확도:   {uncal_acc:.1f}% / 보정: {cal_acc:.1f}%')
print(f'  미보정 cal error: {abs(raw_test.mean()*100 - actual_rate):.1f}%p / 보정: {abs(cal_test.mean()*100 - actual_rate):.1f}%p')

# 확률 구간별 calibration 확인
print('\n구간별 calibration:')
for lo in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    hi = lo + 0.1
    mask = (cal_test >= lo) & (cal_test < hi)
    if mask.sum() < 10:
        continue
    avg_pred = cal_test[mask].mean() * 100
    actual = y_test.values[mask].mean() * 100
    print(f'  {lo*100:.0f}~{hi*100:.0f}%: N={mask.sum():>5}, 평균예측 {avg_pred:.1f}%, 실제 {actual:.1f}%, 차이 {avg_pred-actual:+.1f}%p')

## 10. 모델 Export (서버 호환 포맷)

In [ ]:
import joblib
from datetime import datetime

payload = {
    'model': model,
    'calibrator': calibrator,
    'feature_names': list(X.columns),
    'score': float(cal_acc),
    'train_date': datetime.now().strftime('%Y-%m-%d %H:%M'),
}

ts = datetime.now().strftime('%Y%m%d_%H%M')
model_path = f'/content/xgb_model_minute_{ts}.pkl'
drive_path = f'{MODEL_DIR}/xgb_model_minute_{ts}.pkl'

joblib.dump(payload, model_path)
joblib.dump(payload, drive_path)

print(f'✅ 저장 완료')
print(f'  Colab local: {model_path}')
print(f'  Drive 백업:  {drive_path}')
print(f'\n📦 payload keys: {list(payload.keys())}')
print(f'  피처 개수: {len(payload["feature_names"])}')
print(f'  정확도: {payload["score"]:.2f}%')
print(f'  학습일: {payload["train_date"]}')

## 11. 다운로드

In [ ]:
from google.colab import files
files.download(model_path)

---

## 12. 서버 적용 가이드

다운된 `xgb_model_minute_YYYYMMDD_HHMM.pkl`을 서버에 적용:

### A. 파일 업로드 (Windows / Mac 동일)
```bash
scp -i "F:\Downloads\coinmate.pem" xgb_model_minute_YYYYMMDD_HHMM.pkl \
  ec2-user@43.201.190.36:/home/ec2-user/CoinMate-Backend/models/xgb_model.pkl
```

### B. 서버 코드 변경 (Claude에게 요청)
**중요**: 이 모델은 분봉 기반이라 서버의 `MLPredictor.build_features()`도 위 셀 5의 함수와 **동일하게** 교체되어야 함. 또한 추론 시 `cached_min_dfs`(현재 1시간봉)가 아닌 `minute5` 데이터를 fetch해야 함.

Claude에 다음을 요청:
> "Colab에서 학습한 분봉 모델을 적용해줘. ml_predictor.build_features를 노트북 셀 5와 동일하게 교체하고, get_smart_candles가 minute5 데이터도 캐시하도록 수정해줘."

### C. 검증
```bash
ssh -i "F:\Downloads\coinmate.pem" ec2-user@43.201.190.36
sudo systemctl restart coinmate
# 로그 확인
sudo journalctl -u coinmate --since '20 sec ago' | grep -i ML
# 예상: '>>> 🤖 [ML] 모델 로드 완료 (정확도: XX.X%, 학습일: ... + calibrated)'
```

---

## 13. 재학습 사이클

월 1회 또는 분기 1회 재학습 권장:

1. 이 노트북 재실행 (캐시 덕에 데이터 다운은 신규 분만)
2. 셀 5~10 순서로 실행
3. 새 모델 다운 → 서버 업로드
4. 서버 재시작

💡 Drive에 `xgb_model_minute_*.pkl` 누적 백업되므로 롤백 가능.